In [ ]:

# Step 4: CNN -> BiGRU -> CTC baseline

import json, math, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn

import experiment_config as cfg
from py import train_utils
from py.bopomofo_utils import encode_bopomofo_ids
from py.loss import ctc_loss
from py.evaluation import evaluate_train_val, evaluate_test, predict_bopomofo_ids
from py.models import Baseline, TPL

torch.manual_seed(cfg.SEED); np.random.seed(cfg.SEED); random.seed(cfg.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")

In [ ]:

# Load the files created in Steps 1-3.
#
# X_npy: log-mel spectrograms for all speech samples
#   shape = (N, 1, F, T_MAX)
#   N = number of speech samples
#   F = number of mel-frequency bins (= 64)
#   T_MAX = maximum number of time frames (= 626 with the current preprocessing)
#
# meta: information for each speech sample
#   sentence         = original Chinese sentence
#   bopomofo_tokens  = the same sentence converted to Bopomofo syllables in Steps 2-3
#   valid_max_index  = last valid time-frame index before the padded part of X
#
# vocab:
#   token_to_id: Bopomofo syllable+tone -> class ID
#   id_to_token: class ID -> Bopomofo syllable+tone
#   blank_id   : ID reserved for the CTC blank symbol

X_npy = np.load(cfg.X_PATH)                  # read X.npy
meta = pd.read_csv(cfg.META_PATH)            # read meta.csv
with open(cfg.VOCAB_PATH, encoding="utf-8") as f: vocab = json.load(f)

token_to_id = vocab["token_to_id"]
id_to_token = vocab["id_to_token"]
blank_id = int(vocab["blank_id"])
vocab_size = len(id_to_token)

N, _, F, T_MAX = X_npy.shape
print("X:", X_npy.shape, "| samples:", N, "| vocab:", vocab_size)

In [ ]:

# Convert the correct Bopomofo syllables of each speech sample into vocab IDs for CTC training.
#
# Data flow for one sample:
#   sentence  ->  bopomofo_tokens (already created in Steps 2-3)  ->  Bopomofo vocab IDs
#
# bopomofo_id_lists:
#   one ID sequence for each speech sample
#
# bopomofo_lengths_npy:
#   number of correct Bopomofo syllables in each speech sample
#
# bopomofo_ids_padded_npy:
#   the ID sequences above, padded to the same length so they can be stored as one 2D array
#   padding is only for storage/batching; it is removed before CTCLoss is computed
#
# cnn_time_lengths_npy:
#   valid time length of each speech sample after the CNN reduces the time axis by T/4

bopomofo_id_lists, bopomofo_lengths_npy, bopomofo_ids_padded_npy = \
    encode_bopomofo_ids(meta, token_to_id, blank_id)

cnn_time_lengths_npy = (meta["valid_max_index"].to_numpy(np.int64) + 1) // cfg.TIME_DOWNSAMPLE

# Show the actual conversion for one sample instead of an artificial example.
i = 0
print("sentence :", meta.loc[i, "sentence"])
print("bopomofo :", meta.loc[i, "bopomofo_tokens"])
print("ID list  :", bopomofo_id_lists[i])
print("length   :", bopomofo_lengths_npy[i])

In [ ]:

# Move the full dataset to GPU once.
#
# X:
#   log-mel spectrograms, shape = (N, 1, F, T_MAX), F = 64
#
# bopomofo_ids_padded:
#   correct Bopomofo vocab IDs after padding, shape = (N, longest Bopomofo sentence length)
#   the padding values are NOT treated as correct labels by CTCLoss
#
# bopomofo_lengths:
#   number of correct Bopomofo syllables in each speech sample
#
# cnn_time_lengths:
#   valid CNN output length for each speech sample after the time axis is reduced by T/4

X = torch.from_numpy(X_npy).to(device, dtype=torch.float32)
bopomofo_ids_padded = torch.from_numpy(bopomofo_ids_padded_npy).to(device)
bopomofo_lengths = torch.from_numpy(bopomofo_lengths_npy).to(device)
cnn_time_lengths = torch.from_numpy(cnn_time_lengths_npy).to(device)

print("X:", X.shape)
print("Bopomofo ID matrix:", bopomofo_ids_padded.shape)

In [ ]:

# Randomly split all speech samples into train / validation / test = 85% / 5% / 10%.
# Example: if sample 123 is in train_idx, then X[123] and meta.loc[123] belong to the training set.

g = torch.Generator(device=device).manual_seed(cfg.SEED)
perm = torch.randperm(N, generator=g, device=device)

n_train, n_val = int(N * cfg.TRAIN_FRAC), int(N * cfg.VAL_FRAC)
train_idx = perm[:n_train]
val_idx = perm[n_train:n_train + n_val]
test_idx = perm[n_train + n_val:]

print("train / val / test:", len(train_idx), len(val_idx), len(test_idx))

In [ ]:

# Select Baseline or TPL.
#
# The final dimension-reduction step differs between the two models:
#   baseline : MaxPool(4,1), (c3, F/4, T/4) -> (c3, F/16, T/4)
#   tpl      : TensorProjection, (c3, F/4, T/4) -> (c3/2, F/8, T/4)
#
# The flattened feature size passed to the BiGRU is the same in both models.

c1, c2, c3 = cfg.CNN_CHANNELS

model_specs = {
    "baseline": (Baseline, (F, vocab_size, c1, c2, c3,
                            cfg.GRU_HIDDEN, cfg.GRU_LAYERS, cfg.DROPOUT)),
    "tpl": (TPL, (F, T_MAX, vocab_size, c1, c2, c3,
                  cfg.GRU_HIDDEN, cfg.GRU_LAYERS, cfg.DROPOUT)),
}

model_class, model_args = model_specs[cfg.MODEL]
model = model_class(*model_args).to(device)

print("model:", cfg.MODEL)
print(model)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:

# Training objects.
#
# X and all model parameters are float32.
# CTCLoss receives the model output, the correct Bopomofo IDs,
# the valid CNN output lengths, and the correct Bopomofo lengths.

criterion = nn.CTCLoss(blank=blank_id, reduction="mean", zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)

# These wrappers only avoid repeating the same long argument lists below.
def loss_on(idx):
    logits = model(X[idx])
    return ctc_loss(logits, idx, bopomofo_ids_padded, bopomofo_lengths,
                    cnn_time_lengths, criterion, device)

def train_val_metrics(idx, compute_loss=True):
    return evaluate_train_val(
        idx, model, X, bopomofo_id_lists, bopomofo_ids_padded, bopomofo_lengths,
        cnn_time_lengths, blank_id, criterion, device, cfg.BATCH_SIZE, compute_loss
    )

def predict(idx):
    return predict_bopomofo_ids(
        idx, model, X, cnn_time_lengths, blank_id, cfg.BATCH_SIZE
    )

In [ ]:

# Show the same two validation samples after every epoch.
# This is only for visual inspection; early stopping uses validation CTC loss.

preview_idx = val_idx[:2]

def show_validation_preview():
    for sample_i, pred_ids in zip(preview_idx.cpu().tolist(), predict(preview_idx)):
        print("-" * 80)
        print("Correct sentence:", meta.loc[sample_i, "sentence"])
        print("Correct Bopomofo:", " ".join(id_to_token[k] for k in bopomofo_id_lists[sample_i]))
        print("Prediction      :", " ".join(id_to_token[k] for k in pred_ids))
    print("-" * 80)

In [ ]:

# Train the model. Early stopping is based only on validation CTC loss.

training_start = time.time()
best_val_loss, best_state, bad_epochs, history = math.inf, None, 0, []

for epoch in range(1, cfg.MAX_EPOCHS + 1):
    t0 = time.time()

    # Shuffle the training samples for this epoch.
    model.train()
    order = train_idx[torch.randperm(len(train_idx), device=device)]

    # Train one mini-batch at a time.
    running_loss = 0.0
    for start in range(0, len(order), cfg.BATCH_SIZE):
        idx = order[start:start + cfg.BATCH_SIZE]

        optimizer.zero_grad(set_to_none=True)
        loss = loss_on(idx)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        running_loss += float(loss.detach()) * len(idx)

    train_loss = running_loss / len(train_idx)

    # Measure syllable accuracy and exact-sentence accuracy on train and validation data.
    _, train_ser, train_syllable_acc, train_exact_acc = train_val_metrics(train_idx, compute_loss=False)
    val_loss, val_ser, val_syllable_acc, val_exact_acc = train_val_metrics(val_idx)

    # Save and print this epoch's results.
    train_metrics = (train_ser, train_syllable_acc, train_exact_acc)
    val_metrics = (val_loss, val_ser, val_syllable_acc, val_exact_acc)
    row = train_utils.make_history_row(epoch, train_loss, train_metrics, val_metrics, time.time() - t0)

    history.append(row)
    train_utils.print_epoch(row)
    show_validation_preview()

    # Keep the model with the lowest validation loss.
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = train_utils.copy_model_state(model)
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= cfg.PATIENCE:
            print(f"early stopping (patience={cfg.PATIENCE})")
            break

stopped_epoch = epoch
training_time = time.time() - training_start

history = pd.DataFrame(history)
display(history)

In [ ]:

# Restore the model from the epoch with the lowest validation CTC loss.

model.load_state_dict(best_state)
model.eval()
print("best validation loss:", best_val_loss)

In [ ]:

# Final held-out TEST evaluation.
#
# Strict metrics:
#   prediction must match the G2PW-generated Bopomofo syllable exactly
#
# TW metrics:
#   predefined Taiwan-pronunciation equivalents are also counted as matches
#   tone must still match

test_results = evaluate_test(test_idx, model, X, bopomofo_id_lists, bopomofo_ids_padded, bopomofo_lengths,
                             cnn_time_lengths, blank_id, criterion, id_to_token, device, cfg.BATCH_SIZE)

test_loss, test_ser, test_syllable_acc, test_exact_acc, test_tw_ser, test_tw_syllable_acc, test_tw_exact_acc = test_results

print("FINAL TEST RESULTS")
print("=" * 50)
print(f"CTC loss            : {test_loss:.4f}\n")
print(f"Strict SER          : {test_ser:.4f}")
print(f"Strict syllable acc : {test_syllable_acc:.4f}")
print(f"Strict exact acc    : {test_exact_acc:.4f}\n")
print(f"TW-SER              : {test_tw_ser:.4f}")
print(f"TW-syllable acc     : {test_tw_syllable_acc:.4f}")
print(f"TW-exact acc        : {test_tw_exact_acc:.4f}")

In [ ]:

# Inspect 20 held-out test samples: original sentence, correct Bopomofo, and model prediction.

test_predictions = predict(test_idx)

for i, pred_ids in zip(test_idx.cpu().tolist()[:20], test_predictions[:20]):
    print("=" * 80)
    print("sentence          :", meta.loc[i, "sentence"])
    print("correct Bopomofo  :", " ".join(id_to_token[j] for j in bopomofo_id_lists[i]))
    print("predicted Bopomofo:", " ".join(id_to_token[j] for j in pred_ids))

In [ ]:

# Save the results of this run.
#
# results/baseline/seed_<SEED>/
#   config.json : settings actually used in this run
#   history.csv : epoch-by-epoch training / validation results
#   test.csv    : final held-out test loss and accuracies
#   model.pt    : best model selected by validation CTC loss

RESULT_DIR = Path("../results") / cfg.MODEL / f"seed_{cfg.SEED}"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Snapshot of the settings actually used at runtime.
run_config = {
    "seed": cfg.SEED,
    "model": cfg.MODEL,
    "batch_size": cfg.BATCH_SIZE,
    "max_epochs": cfg.MAX_EPOCHS,
    "patience": cfg.PATIENCE,
    "lr": cfg.LR,
    "weight_decay": cfg.WEIGHT_DECAY,
    "cnn_channels": list(cfg.CNN_CHANNELS),
    "gru_hidden": cfg.GRU_HIDDEN,
    "gru_layers": cfg.GRU_LAYERS,
    "dropout": cfg.DROPOUT,
    "time_downsample": cfg.TIME_DOWNSAMPLE,
    "freq_downsample": cfg.FREQ_DOWNSAMPLE,
    "train_frac": cfg.TRAIN_FRAC,
    "val_frac": cfg.VAL_FRAC,
    "test_frac": cfg.TEST_FRAC,
    "n_mels": F,
    "t_max": T_MAX,
}

with open(RESULT_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(run_config, f, ensure_ascii=False, indent=2)

# Save only the useful, non-redundant training history columns.
history_columns = [
    "epoch",
    "train_loss", "train_syllable_acc", "train_exact_acc",
    "val_loss", "val_syllable_acc", "val_exact_acc",
    "seconds",
]
history[history_columns].to_csv(RESULT_DIR / "history.csv", index=False)

# Final test results. SER is omitted because syllable_acc = 1 - SER.
test = pd.DataFrame([{
    "test_loss": test_loss,
    "strict_syllable_acc": test_syllable_acc,
    "strict_exact_acc": test_exact_acc,
    "tw_syllable_acc": test_tw_syllable_acc,
    "tw_exact_acc": test_tw_exact_acc,
}])
test.to_csv(RESULT_DIR / "test.csv", index=False)

# Save the best model restored after early stopping.
best_epoch = int(history.loc[history["val_loss"].idxmin(), "epoch"])

torch.save({
    "model_state_dict": model.state_dict(),
    "vocab": vocab,
    "seed": cfg.SEED,
    "model": cfg.MODEL,
    "best_epoch": best_epoch,
    "stopped_epoch": int(stopped_epoch),
    "best_val_loss": float(best_val_loss),
    "training_time": float(training_time),
}, RESULT_DIR / "model.pt")

print("saved results:", RESULT_DIR)